# Interaction Lab: uploaded pilot on a qualified free GPU

This notebook uses the private `server-pilot-v1-inputs.zip` prepared on Windows. It verifies the independent archive SHA-256 and every indexed source/cache member, rejects unsafe paths and extra partitions, then atomically stages only train/selection inputs. The dataset is not downloaded again.

Choose an allocated GPU in the platform UI. The kernel may differ from the training runtime: set `QUALIFIED_PYTHON` to an isolated **Python 3.12 + PyTorch 2.13.0 CUDA** environment containing NumPy. A Colab T4 can use its separately qualified CUDA build; a Kaggle P100 needs a build retaining Pascal support. The script fails instead of silently using another stack.

The script freezes one infrastructure pilot epoch with microbatch 1, accumulation 32 and FP32. It reserves and charges qualification, training and selection against a separate **1,800-second** cap. It leaves the two frozen studies and their budgets untouched. A new cloud root has one owner. Do not use this notebook as an SSH service, polling worker, quota workaround or keepalive. [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Kaggle guidance](https://www.kaggle.com/docs/efficient-gpu-usage).

After the cell returns, download both the printed ZIP and receipt through the normal file/output UI. On Windows, verify all hashes and strict checkpoint loading before acknowledging deletion of the export. Keep active checkpoints and original inputs. Results are development infrastructure evidence, with no calibration or locked-holdout inspection.

**Kaggle extracted uploads:** set `EXTRACTED_INPUT_PATH` to the dataset folder containing `package-files.json`, and use `/kaggle/working` paths for `JOB_PATH` and `SCRIPT_PATH`. This mode pins the original ZIP checksum index independently, verifies all 1,076 indexed files and exact folder membership, and ignores only a top-level `cloud-pilot-cell.py` sibling. Export receipts retain the original payload ZIP SHA-256 and explicitly identify verification of indexed extracted members; they do not claim the ZIP itself was present on Kaggle.


In [ ]:
from pathlib import Path
import hashlib
import subprocess

# This interpreter is separate from the notebook kernel, if necessary.
QUALIFIED_PYTHON = Path('/content/interaction-lab-cloud-env/bin/python')
PAYLOAD_PATH = Path('/content/server-pilot-v1-inputs.zip')
# Kaggle extracts the uploaded ZIP: set this to the folder containing package-files.json.
EXTRACTED_INPUT_PATH = None  # e.g. Path('/kaggle/input/datasets/your-account/interaction-lab-pilot')
JOB_PATH = Path('/content/interaction-lab-colab-pilot-v1')
SCRIPT_PATH = Path('/content/interaction-lab-cloud-pilot.py')
EXPECTED_SCRIPT_SHA256 = '00e3b9b943d8ef456ee62485e6dbdf620ff8e5c0cca57748695441c9a4596e85'
SCRIPT_SOURCE = '"""Paste this entire cell into an interactive cloud notebook, or run it there.\nOnly the uploaded, hash-pinned train/selection pilot is used. No account secrets.\n"""\nfrom pathlib import Path, PurePosixPath\nimport hashlib\nimport json\nimport os\nimport shutil\nimport stat\nimport sys\nimport time\nimport uuid\nimport zipfile\n\nPAYLOAD = Path(\'/content/server-pilot-v1-inputs.zip\')\nJOB_ROOT = Path(\'/content/interaction-lab-colab-pilot-v1\')\nEXPECTED_PAYLOAD_SHA256 = \'f3ce396b8bbfa20b62edc6cbf7ccb42c22ae5ac191dd5e956ab46595a226c9fa\'\nEXPECTED_PACKAGE_INDEX_SHA256 = \'3770e3f2c6df73ce4fb652a6c52a2841772f25feb70a3480bb035e788f107282\'\nEXPECTED_INDEXED_MEMBER_COUNT = 1076\nEXTRACTED_INPUT = None\nINPUT_VERIFICATION = None\nMAXIMUM_COMPUTE_SECONDS = 1800.0\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef verify_and_stage(payload, destination, expected_sha256):\n    payload, destination = Path(payload), Path(destination)\n    if payload.is_symlink() or not payload.is_file() or payload.stat().st_size > 300 * 1024**2:\n        raise ValueError(\'Missing/unsafe payload: upload the original private inputs ZIP and set PAYLOAD.\')\n    if sha256_file(payload) != expected_sha256:\n        raise ValueError(\'Uploaded payload SHA-256 differs from the independently pinned local file.\')\n    if destination.is_symlink():\n        raise ValueError(\'Job destination cannot be a symlink.\')\n    destination = destination.resolve()\n    with zipfile.ZipFile(payload) as archive:\n        entries = archive.infolist()\n        names = [entry.filename for entry in entries]\n        if len(entries) > 2000 or len(set(names)) != len(names) or sum(e.file_size for e in entries) > 512 * 1024**2:\n            raise ValueError(\'Payload bounds or duplicate-name checks failed.\')\n        for entry in entries:\n            name, parts = entry.filename, PurePosixPath(entry.filename).parts\n            if (\'\\\\\' in name or \':\' in name or name.startswith(\'/\') or \'..\' in parts\n                    or PurePosixPath(name).as_posix() != name or entry.is_dir()\n                    or stat.S_ISLNK(entry.external_attr >> 16) or entry.file_size > 8 * 1024**2):\n                raise ValueError(\'Unsafe payload member: \' + name)\n            allowed = name in {\'package-files.json\', \'initial-config.json\', \'source-manifest.json\',\n                               \'cache/manifest.json\', \'cache/preprocessing.json\'}\n            allowed |= len(parts) == 3 and parts[:2] == (\'source\', \'adp\') and name.endswith(\'.py\')\n            allowed |= len(parts) == 3 and parts[0] == \'cache\' and parts[1] in {\'train\', \'validation\'} and name.endswith(\'.npz\')\n            if not allowed:\n                raise ValueError(\'Payload contains an unexpected file/partition: \' + name)\n        if \'package-files.json\' not in names:\n            raise ValueError(\'Missing payload checksum index.\')\n        index = json.loads(archive.read(\'package-files.json\'))\n        if not isinstance(index, dict) or set(index) != set(names) - {\'package-files.json\'}:\n            raise ValueError(\'Payload files differ from their checksum index.\')\n        for name, info in index.items():\n            data = archive.read(name)\n            if not isinstance(info, dict) or len(data) != info.get(\'size\') or hashlib.sha256(data).hexdigest() != info.get(\'sha256\'):\n                raise ValueError(\'Payload member checksum differs: \' + name)\n        manifest = json.loads(archive.read(\'cache/manifest.json\'))\n        if set(manifest.get(\'splits\', {})) != {\'train\', \'validation\'}:\n            raise ValueError(\'Pilot requires train and selection only.\')\n        if destination.exists():\n            for name, info in index.items():\n                target = destination / name\n                if target.is_symlink() or not target.is_file() or sha256_file(target) != info[\'sha256\']:\n                    raise ValueError(\'Existing job inputs differ; choose a fresh JOB_ROOT.\')\n            return destination\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        if shutil.disk_usage(destination.parent).free - sum(e.file_size for e in entries) - 128 * 1024**2 < 8 * 1024**3:\n            raise RuntimeError(\'Insufficient local runtime storage; preserve 8 GiB.\')\n        staging = destination.with_name(destination.name + \'.staging-\' + uuid.uuid4().hex)\n        staging.mkdir()\n        for entry in entries:\n            target = staging / entry.filename\n            target.parent.mkdir(parents=True, exist_ok=True)\n            with target.open(\'xb\') as stream:\n                stream.write(archive.read(entry.filename))\n        staging.rename(destination)\n    return destination\n\n\ndef verify_extracted_and_stage(source, destination, expected_index_sha256=EXPECTED_PACKAGE_INDEX_SHA256,\n                               expected_member_count=EXPECTED_INDEXED_MEMBER_COUNT):\n    source, destination = Path(source), Path(destination)\n    linked = lambda path: path.is_symlink() or (hasattr(path, \'is_junction\') and path.is_junction())\n    if linked(source) or not source.is_dir() or linked(destination):\n        raise ValueError(\'Extracted input and job destination must be real directories, not links.\')\n    source, destination = source.resolve(), destination.resolve()\n    if source == destination or source in destination.parents or destination in source.parents:\n        raise ValueError(\'Extracted inputs and job destination must be separate directories.\')\n    index_path = source/\'package-files.json\'\n    if linked(index_path) or not index_path.is_file() or index_path.stat().st_size > 2*1024**2:\n        raise ValueError(\'Missing or excessive extracted package checksum index.\')\n    index_bytes = index_path.read_bytes()\n    if hashlib.sha256(index_bytes).hexdigest() != expected_index_sha256:\n        raise ValueError(\'Extracted package index SHA-256 differs from the independently pinned original ZIP index.\')\n    index = json.loads(index_bytes)\n    if not isinstance(index, dict) or len(index) != expected_member_count or len({name.casefold() for name in index}) != len(index):\n        raise ValueError(\'Extracted package index has an unexpected indexed member count or duplicate paths.\')\n    total, directories = len(index_bytes), set()\n    for name, info in index.items():\n        parts = PurePosixPath(name).parts\n        allowed = name in {\'initial-config.json\', \'source-manifest.json\', \'cache/manifest.json\', \'cache/preprocessing.json\'}\n        allowed |= len(parts) == 3 and parts[:2] == (\'source\', \'adp\') and name.endswith(\'.py\')\n        allowed |= len(parts) == 3 and parts[0] == \'cache\' and parts[1] in {\'train\', \'validation\'} and name.endswith(\'.npz\')\n        if (\'\\\\\' in name or \':\' in name or name.startswith(\'/\') or \'..\' in parts\n                or PurePosixPath(name).as_posix() != name or not allowed):\n            raise ValueError(\'Unsafe or unexpected extracted input path: \'+name)\n        if (not isinstance(info, dict) or set(info) != {\'size\', \'sha256\'} or type(info[\'size\']) is not int\n                or not 0 <= info[\'size\'] <= 8*1024**2 or not isinstance(info[\'sha256\'], str)\n                or len(info[\'sha256\']) != 64 or any(c not in \'0123456789abcdef\' for c in info[\'sha256\'])):\n            raise ValueError(\'Invalid extracted package checksum entry: \'+name)\n        total += info[\'size\']\n        directories.update(PurePosixPath(*parts[:level]).as_posix() for level in range(1, len(parts)))\n    if total > 512*1024**2:\n        raise ValueError(\'Extracted payload exceeds the expanded size limit.\')\n    actual = set()\n    for folder, folders, files in os.walk(source, followlinks=False):\n        for name in folders+files:\n            item = Path(folder)/name\n            if linked(item):\n                raise ValueError(\'Extracted payload cannot contain filesystem links.\')\n        if any((Path(folder)/name).relative_to(source).as_posix() not in directories for name in folders):\n            raise ValueError(\'Extracted payload contains an unexpected directory.\')\n        for name in files:\n            item = Path(folder)/name\n            relative = item.relative_to(source).as_posix()\n            if not item.is_file() or len(actual) >= expected_member_count+2:\n                raise ValueError(\'Extracted payload contains a nonregular file or too many files.\')\n            actual.add(relative)\n    required = set(index) | {\'package-files.json\'}\n    if actual - {\'cloud-pilot-cell.py\'} != required:\n        raise ValueError(\'Extracted payload files differ from the pinned index; only a sibling cloud-pilot-cell.py may be ignored.\')\n    if \'cloud-pilot-cell.py\' in actual and (source/\'cloud-pilot-cell.py\').stat().st_size > 128*1024:\n        raise ValueError(\'Sibling pilot runner exceeds the supported size.\')\n    for name, info in index.items():\n        item = source/name\n        if item.stat().st_size != info[\'size\'] or sha256_file(item) != info[\'sha256\']:\n            raise ValueError(\'Extracted payload member checksum differs: \'+name)\n    manifest = json.loads((source/\'cache/manifest.json\').read_bytes())\n    if set(manifest.get(\'splits\', {})) != {\'train\', \'validation\'}:\n        raise ValueError(\'Pilot requires train and selection only.\')\n    if destination.exists():\n        if (not destination.is_dir() or linked(destination/\'package-files.json\')\n                or not (destination/\'package-files.json\').is_file()\n                or any(linked(destination/name) for name in directories)\n                or sha256_file(destination/\'package-files.json\') != expected_index_sha256):\n            raise ValueError(\'Existing job inputs differ; choose a fresh JOB_ROOT.\')\n        for name, info in index.items():\n            target = destination/name\n            if linked(target) or not target.is_file() or target.stat().st_size != info[\'size\'] or sha256_file(target) != info[\'sha256\']:\n                raise ValueError(\'Existing job inputs differ; choose a fresh JOB_ROOT.\')\n        return destination\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if shutil.disk_usage(destination.parent).free-total-128*1024**2 < 8*1024**3:\n        raise RuntimeError(\'Insufficient local runtime storage; preserve 8 GiB.\')\n    staging = destination.with_name(destination.name+\'.staging-\'+uuid.uuid4().hex)\n    staging.mkdir()\n    try:\n        (staging/\'package-files.json\').write_bytes(index_bytes)\n        for name, info in index.items():\n            with (source/name).open(\'rb\') as stream:\n                data = stream.read(info[\'size\']+1)\n            if len(data) != info[\'size\'] or hashlib.sha256(data).hexdigest() != info[\'sha256\']:\n                raise ValueError(\'Extracted input changed during atomic staging: \'+name)\n            target = staging/name\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(data)\n        staging.rename(destination)\n    finally:\n        if staging.exists():\n            assert staging.resolve().parent == destination.parent\n            shutil.rmtree(staging)\n    return destination\n\n\ndef export_pilot(root):\n    root = Path(root)\n    if (root / \'.remote-job.lock\').exists():\n        raise RuntimeError(\'Cannot export while a training owner is active.\')\n    candidates = [\'job.json\', \'config.json\', \'compute-ledger.json\', \'status.json\', \'environment.json\',\n                  \'qualification.json\', \'package-files.json\', \'initial-config.json\',\n                  \'cache/manifest.json\', \'cache/preprocessing.json\',\n                  \'run/last.pt\', \'run/best.pt\', \'run/history.json\']\n    files = [name for name in candidates if (root / name).is_file()]\n    if \'run/last.pt\' not in files:\n        raise RuntimeError(\'No resumable checkpoint yet; inspect status.json and qualification.json.\')\n    if any((root / name).is_symlink() for name in files):\n        raise ValueError(\'Linked export files are not accepted.\')\n    exports = root / \'exports\'\n    exports.mkdir(exist_ok=True)\n    if shutil.disk_usage(exports).free - sum((root / n).stat().st_size for n in files) < 8 * 1024**3:\n        raise RuntimeError(\'Insufficient space for verified export and 8 GiB reserve.\')\n    tag = time.strftime(\'%Y%m%dT%H%M%SZ\', time.gmtime()) + \'-\' + uuid.uuid4().hex[:12]\n    partial, target = exports / (tag + \'.zip.partial\'), exports / (tag + \'.zip\')\n    receipt = {\'version\': 1, \'purpose\': \'standalone cloud infrastructure pilot, not frozen-study evidence\',\n               \'payload_sha256\': EXPECTED_PAYLOAD_SHA256,\n               \'files\': {name: {\'sha256\': sha256_file(root/name), \'bytes\': (root/name).stat().st_size} for name in files}}\n    if INPUT_VERIFICATION is not None:\n        receipt[\'input_verification\'] = INPUT_VERIFICATION\n    with zipfile.ZipFile(partial, \'w\', compression=zipfile.ZIP_STORED) as archive:\n        for name in files:\n            archive.write(root/name, name)\n        archive.writestr(\'export-manifest.json\', json.dumps(receipt, indent=2)+\'\\n\')\n    with zipfile.ZipFile(partial) as archive:\n        for name, info in receipt[\'files\'].items():\n            if hashlib.sha256(archive.read(name)).hexdigest() != info[\'sha256\']:\n                raise OSError(\'Export checksum mismatch; partial archive retained.\')\n    partial.rename(target)\n    receipt[\'archive_sha256\'] = sha256_file(target)\n    receipt_path = target.with_suffix(\'.receipt.json\')\n    receipt_path.write_text(json.dumps(receipt, indent=2)+\'\\n\', encoding=\'utf-8\')\n    print(\'DOWNLOAD BOTH FILES; keep the active checkpoints:\', target, receipt_path, sep=\'\\n\', flush=True)\n    return target, receipt_path\n\n\ndef main():\n    global INPUT_VERIFICATION\n    INPUT_VERIFICATION = None\n    assert sys.version_info[:2] == (3, 12), \'This pilot requires Python 3.12. Select a matching notebook runtime.\'\n    import torch\n    assert str(torch.__version__).split(\'+\')[0] == \'2.13.0\', (\'PyTorch 2.13.0 qualification is required; current runtime is \' + str(torch.__version__) + \'. Stop and install an official compatible CUDA build, then restart the notebook runtime.\')\n    assert torch.cuda.is_available(), \'No CUDA GPU allocated; select a free accelerator in the notebook UI.\'\n    os.environ[\'OMP_NUM_THREADS\'] = os.environ[\'MKL_NUM_THREADS\'] = \'2\'\n    torch.set_num_threads(2)\n    if EXTRACTED_INPUT is None:\n        root = verify_and_stage(PAYLOAD, JOB_ROOT, EXPECTED_PAYLOAD_SHA256)\n    else:\n        root = verify_extracted_and_stage(EXTRACTED_INPUT, JOB_ROOT)\n        INPUT_VERIFICATION = {\'kind\': \'verified_indexed_extracted_members\',\n            \'package_index_sha256\': EXPECTED_PACKAGE_INDEX_SHA256,\n            \'verified_member_count\': EXPECTED_INDEXED_MEMBER_COUNT}\n        print(\'Input verification: verified indexed extracted members from the pinned original ZIP index.\', flush=True)\n    sys.path.insert(0, str(root/\'source\'))\n    from adp.data import CachedDataset, collate_scenes, read_json, write_json\n    from adp.model import ForecastModel, forecast_loss, model_inputs\n    from adp.remote_jobs import prepare_job, run_job, _owner_lock\n    from adp.training import environment_report, gpu_smoke_test, _smoke_batch\n    config = read_json(root/\'initial-config.json\')\n    protocol = prepare_job(root, config, epochs=1, max_runtime_seconds=MAXIMUM_COMPUTE_SECONDS,\n                           source_root=root/\'source/adp\')\n    environment = environment_report()\n    write_json(root/\'environment.json\', environment)\n    print(json.dumps(environment, indent=2), flush=True)\n    ledger_path = root/\'compute-ledger.json\'\n    with _owner_lock(root):\n        ledger = read_json(ledger_path) if ledger_path.exists() else {\'version\': 1, \'protocol_sha256\': protocol[\'protocol_sha256\'], \'charged_seconds\': 0., \'slices\': []}\n        assert ledger[\'protocol_sha256\'] == protocol[\'protocol_sha256\']\n        available = MAXIMUM_COMPUTE_SECONDS - ledger[\'charged_seconds\']\n        if available <= 0:\n            raise RuntimeError(\'Pilot budget exhausted; do not reset the ledger.\')\n        entry = {\'started_unix\': time.time(), \'device\': \'cuda\', \'state\': \'qualification_reserved\',\n                 \'reserved_seconds\': available, \'charged_seconds\': available}\n        ledger[\'charged_seconds\'] += available\n        ledger[\'slices\'].append(entry)\n        write_json(ledger_path, ledger)\n        started = time.monotonic()\n        qualification = {\'purpose\': protocol[\'purpose\'], \'protocol_sha256\': protocol[\'protocol_sha256\']}\n        try:\n            qualification[\'kernel_model_smoke\'] = gpu_smoke_test(\'cuda\', config)\n            # Full actor/map shapes, microbatch one, FP32 as frozen by prepare_job.\n            batch = _smoke_batch(torch.device(\'cuda\'))\n            actor_keys = {\'history\', \'history_mask\', \'actor_mask\', \'dynamic_mask\', \'actor_types\', \'future\', \'future_mask\'}\n            map_keys = {\'map_points\', \'map_features\', \'map_mask\'}\n            for key, value in list(batch.items()):\n                repeat = [1]*value.ndim\n                if key in actor_keys: repeat[1] = 16\n                if key in map_keys: repeat[1] = 128\n                batch[key] = value.repeat(*repeat)\n            model = ForecastModel(config[\'model\']).cuda().train()\n            optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)\n            torch.cuda.reset_peak_memory_stats()\n            output = model(**model_inputs(batch))\n            loss = forecast_loss(output, batch)[\'loss\']\n            loss.backward()\n            assert output[\'means\'].shape == (1, 6, 32, 60, 2)\n            assert output[\'scales\'].min() >= .15 and output[\'scales\'].max() <= 20\n            torch.testing.assert_close(output[\'logits\'].softmax(-1).sum(-1), torch.ones(1, device=\'cuda\'))\n            assert torch.isfinite(loss) and all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)\n            optimizer.step()\n            torch.cuda.synchronize()\n            qualification[\'full_shape_fp32\'] = {\'actors\': 32, \'map_polylines\': 128,\n                \'microbatch\': 1, \'loss\': float(loss.detach()), \'peak_allocated_bytes\': torch.cuda.max_memory_allocated()}\n            model.eval()\n            cpu_model = ForecastModel(config[\'model\']).eval()\n            cpu_model.load_state_dict({key: value.detach().cpu() for key, value in model.state_dict().items()})\n            cpu_inputs = {key: value.cpu() for key, value in model_inputs(batch).items()}\n            with torch.inference_mode():\n                expected, actual = cpu_model(**cpu_inputs), model(**model_inputs(batch))\n            for key in (\'means\', \'scales\', \'logits\', \'risk_logits\'):\n                torch.testing.assert_close(actual[key].cpu(), expected[key], rtol=1e-4, atol=1e-4)\n            qualification[\'cpu_gpu_agreement\'] = \'passed\'\n            del model, cpu_model, optimizer, batch, output, loss, expected, actual, cpu_inputs\n            torch.cuda.empty_cache()\n            # Real data qualification reads one TRAIN scenario, never selection/holdout labels here.\n            scene = CachedDataset(root/\'cache\', \'train\')[0]\n            batch = {key: value.cuda() if isinstance(value, torch.Tensor) else value for key, value in collate_scenes([scene]).items()}\n            model = ForecastModel(config[\'model\']).cuda().train()\n            optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)\n            torch.cuda.synchronize()\n            measured = time.perf_counter()\n            output = model(**model_inputs(batch))\n            loss = forecast_loss(output, batch)[\'loss\']\n            loss.backward()\n            assert torch.isfinite(loss) and all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)\n            optimizer.step()\n            torch.cuda.synchronize()\n            qualification[\'real_train_step\'] = {\'seconds\': time.perf_counter()-measured, \'loss\': float(loss.detach()), \'scenarios\': 1}\n            del model, optimizer, batch, output, loss\n            torch.cuda.empty_cache()\n            qualification[\'status\'] = \'passed\'\n        except BaseException as error:\n            qualification.update(status=\'failed\', error=f\'{type(error).__name__}: {error}\')\n            raise\n        finally:\n            elapsed = time.monotonic()-started\n            entry.update(state=\'qualification_\'+qualification.get(\'status\', \'failed\'), charged_seconds=elapsed)\n            ledger[\'charged_seconds\'] += elapsed-available\n            qualification[\'charged_seconds\'] = elapsed\n            write_json(ledger_path, ledger)\n            write_json(root/\'qualification.json\', qualification)\n    print(\'Qualification:\', json.dumps(qualification, indent=2), flush=True)\n    try:\n        result = run_job(root, device=\'cuda\', slice_seconds=MAXIMUM_COMPUTE_SECONDS)\n        print(\'Training result:\', json.dumps(result, indent=2), flush=True)\n    finally:\n        if (root/\'run/last.pt\').is_file():\n            export_pilot(root)\n\n\nif __name__ == \'__main__\':\n    import argparse\n    parser = argparse.ArgumentParser(description=__doc__)\n    inputs = parser.add_mutually_exclusive_group()\n    inputs.add_argument(\'--payload\', type=Path, default=PAYLOAD)\n    inputs.add_argument(\'--extracted-input\', type=Path, help=\'Kaggle-extracted directory containing the original pinned package-files.json\')\n    parser.add_argument(\'--root\', type=Path, default=JOB_ROOT)\n    arguments = parser.parse_args()\n    PAYLOAD, JOB_ROOT, EXTRACTED_INPUT = arguments.payload, arguments.root, arguments.extracted_input\n    main()\n'
assert hashlib.sha256(SCRIPT_SOURCE.encode('utf-8')).hexdigest() == EXPECTED_SCRIPT_SHA256
SCRIPT_PATH.write_text(SCRIPT_SOURCE, encoding='utf-8')
input_args = ['--extracted-input', str(EXTRACTED_INPUT_PATH)] if EXTRACTED_INPUT_PATH is not None else ['--payload', str(PAYLOAD_PATH)]
subprocess.run([str(QUALIFIED_PYTHON), str(SCRIPT_PATH), *input_args, '--root', str(JOB_PATH)], check=True)
